# 3D Resampling with B-Spline Interpolation

This notebook demonstrates the B-spline resampling pipeline used to preprocess
rectal cancer MRI volumes to isotropic spacing (0.36mm) before classification.

This is the same resampling logic used in `prepare_3d_data.py`, presented here
for interactive exploration and verification.

## Key Steps
1. Read original NIfTI volumes
2. Resample XY axes to target spacing (B-spline for image, nearest-neighbor for label)
3. Resample Z axis to target spacing
4. Verify output spacing and dimensions

In [ ]:
import os
import glob
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

## Resampling Functions

In [ ]:
def resample_xyz_axis(im_image, space=(1., 1., 1.), interp=sitk.sitkLinear):
    """Resample image to target spacing along all axes."""
    identity = sitk.Transform(3, sitk.sitkIdentity)
    sp = im_image.GetSpacing()
    sz = im_image.GetSize()

    new_sz = (
        int(round(sz[0] * sp[0] / space[0])),
        int(round(sz[1] * sp[1] / space[1])),
        int(round(sz[2] * sp[2] / space[2])),
    )

    ref = sitk.Image(new_sz, im_image.GetPixelIDValue())
    ref.SetSpacing(space)
    ref.SetOrigin(im_image.GetOrigin())
    ref.SetDirection([1, 0, 0, 0, 1, 0, 0, 0, 1])

    return sitk.Resample(im_image, ref, identity, interp)


def resample_label_to_ref(im_label, im_ref, interp=sitk.sitkNearestNeighbor):
    """Resample label to match reference image (handles multi-label)."""
    identity = sitk.Transform(3, sitk.sitkIdentity)

    ref = sitk.Image(im_ref.GetSize(), im_label.GetPixelIDValue())
    ref.SetSpacing(im_ref.GetSpacing())
    ref.SetOrigin(im_ref.GetOrigin())
    ref.SetDirection(im_ref.GetDirection())

    np_label = sitk.GetArrayFromImage(im_label)
    labels = np.unique(np_label)
    resampled_list = []

    for lbl_val in labels:
        tmp = (np_label == lbl_val).astype(np.uint8)
        tmp_img = sitk.GetImageFromArray(tmp)
        tmp_img.CopyInformation(im_label)
        tmp_resampled = sitk.Resample(tmp_img, ref, identity, interp)
        resampled_list.append(sitk.GetArrayFromImage(tmp_resampled))

    one_hot = np.stack(resampled_list, axis=0)
    result = np.argmax(one_hot, axis=0).astype(np.uint8)
    out = sitk.GetImageFromArray(result)
    out.CopyInformation(im_ref)
    return out

## Configure Paths

In [ ]:
# --- Configure paths ---
DATA_DIR = 'data/3d'       # Update to your data directory
OUTPUT_DIR = 'data/3d_resampled'

TARGET_SPACING = (0.36, 0.36, 0.36)  # Isotropic target spacing (mm)

## Inspect Original Data

In [ ]:
image_dir = os.path.join(DATA_DIR, 'imagesTr')
mask_dir  = os.path.join(DATA_DIR, 'labelsTr')

image_files = sorted(glob.glob(os.path.join(image_dir, '*.nii.gz')))
mask_files  = sorted(glob.glob(os.path.join(mask_dir, '*.nii.gz')))

print(f'Found {len(image_files)} images and {len(mask_files)} labels')

# Inspect first few cases
for img_path, mask_path in zip(image_files[:5], mask_files[:5]):
    name = os.path.basename(img_path).replace('.nii.gz', '')
    img = sitk.ReadImage(img_path)
    mask = sitk.ReadImage(mask_path)
    print(f'{name}: size={img.GetSize()}, spacing={tuple(round(s,4) for s in img.GetSpacing())}')

## Resample Single Case (Interactive)

In [ ]:
# Pick one case to resample interactively
case_idx = 0
img_path = image_files[case_idx]
mask_path = mask_files[case_idx]
name = os.path.basename(img_path).replace('.nii.gz', '')

image = sitk.ReadImage(img_path)
label = sitk.ReadImage(mask_path)

spacing = image.GetSpacing()
print(f'Original: size={image.GetSize()}, spacing={tuple(round(s,4) for s in spacing)}')

# Step 1: Resample XY to target (B-spline for image)
re_img_xy = resample_xyz_axis(image,
    space=(TARGET_SPACING[0], TARGET_SPACING[1], spacing[2]),
    interp=sitk.sitkBSpline)
re_lab_xy = resample_label_to_ref(label, re_img_xy,
    interp=sitk.sitkBSpline)

# Step 2: Resample Z to target
re_img = resample_xyz_axis(re_img_xy,
    space=TARGET_SPACING,
    interp=sitk.sitkBSpline)
re_lab = resample_label_to_ref(re_lab_xy, re_img,
    interp=sitk.sitkBSpline)

print(f'Resampled: size={re_img.GetSize()}, spacing={re_img.GetSpacing()}')
assert re_img.GetSpacing() == re_lab.GetSpacing()

In [ ]:
# Visualize original vs resampled (mid-slice comparison)
orig_np = sitk.GetArrayFromImage(image)
resampled_np = sitk.GetArrayFromImage(re_img)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original
z_n = orig_np.shape[0] // 2
axes[0][0].imshow(orig_np[z_n], cmap='gray')
axes[0][0].set_title(f'Original Axial (z={z_n}, shape={orig_np.shape})')
axes[0][0].axis('off')

y_n = orig_np.shape[1] // 2
axes[0][1].imshow(orig_np[:, y_n, :], cmap='gray')
axes[0][1].set_title(f'Original Coronal (y={y_n})')
axes[0][1].axis('off')

x_n = orig_np.shape[2] // 2
axes[0][2].imshow(orig_np[:, :, x_n], cmap='gray')
axes[0][2].set_title(f'Original Sagittal (x={x_n})')
axes[0][2].axis('off')

# Resampled
z_n = resampled_np.shape[0] // 2
axes[1][0].imshow(resampled_np[z_n], cmap='gray')
axes[1][0].set_title(f'Resampled Axial (z={z_n}, shape={resampled_np.shape})')
axes[1][0].axis('off')

y_n = resampled_np.shape[1] // 2
axes[1][1].imshow(resampled_np[:, y_n, :], cmap='gray')
axes[1][1].set_title(f'Resampled Coronal (y={y_n})')
axes[1][1].axis('off')

x_n = resampled_np.shape[2] // 2
axes[1][2].imshow(resampled_np[:, :, x_n], cmap='gray')
axes[1][2].set_title(f'Resampled Sagittal (x={x_n})')
axes[1][2].axis('off')

plt.suptitle(f'{name}: Original vs Resampled ({TARGET_SPACING}mm)', fontsize=14)
plt.tight_layout()
plt.show()

## Batch Resample All Cases

In [ ]:
# Batch process all cases
img_out_dir = os.path.join(OUTPUT_DIR, 'imagesTr')
mask_out_dir = os.path.join(OUTPUT_DIR, 'labelsTr')
os.makedirs(img_out_dir, exist_ok=True)
os.makedirs(mask_out_dir, exist_ok=True)

for img_path, mask_path in zip(image_files, mask_files):
    name = os.path.basename(img_path).replace('.nii.gz', '')

    image = sitk.ReadImage(img_path)
    label = sitk.ReadImage(mask_path)
    spacing = image.GetSpacing()

    # Resample XY then Z (B-spline interpolation)
    re_img_xy = resample_xyz_axis(image,
        space=(TARGET_SPACING[0], TARGET_SPACING[1], spacing[2]),
        interp=sitk.sitkBSpline)
    re_lab_xy = resample_label_to_ref(label, re_img_xy, interp=sitk.sitkBSpline)

    re_img = resample_xyz_axis(re_img_xy, space=TARGET_SPACING, interp=sitk.sitkBSpline)
    re_lab = resample_label_to_ref(re_lab_xy, re_img, interp=sitk.sitkBSpline)

    sitk.WriteImage(re_img, os.path.join(img_out_dir, f'{name}.nii.gz'))
    sitk.WriteImage(re_lab, os.path.join(mask_out_dir, f'{name}.nii.gz'))

    print(f'{name}: {image.GetSize()} -> {re_img.GetSize()}, spacing={re_img.GetSpacing()}')

print(f'\nDone! Resampled {len(image_files)} cases to {OUTPUT_DIR}')

## Verify Resampled Data

In [ ]:
# Verify all resampled files have correct spacing
resampled_files = sorted(glob.glob(os.path.join(img_out_dir, '*.nii.gz')))

all_ok = True
for f in resampled_files:
    img = sitk.ReadImage(f)
    name = os.path.basename(f).replace('.nii.gz', '')
    sp = img.GetSpacing()
    if sp != TARGET_SPACING:
        print(f'MISMATCH: {name} spacing={sp}')
        all_ok = False

if all_ok:
    print(f'All {len(resampled_files)} cases verified: spacing = {TARGET_SPACING} mm')